# YgoMaster-PackBuilder — Policies Validation Tests

Notebook di test manuale/esplorativo per il **policies validation layer**.

Obiettivi:

- caricare `config/policies.yaml`;
- verificare che la configurazione reale sia valida;
- controllare `ValidationReport`;
- testare una regola alla volta usando copie modificate in memoria;
- verificare probabilità, somme, key e semantica del pity UR.

## 1. Ambiente e root della repository

In [12]:
from pathlib import Path
import sys

print("Python interpreter:", sys.executable)
print("Current working directory:", Path.cwd())

cwd = Path.cwd().resolve()

if (cwd / "pyproject.toml").exists():
    repo_root = cwd
elif (cwd.parent / "pyproject.toml").exists():
    repo_root = cwd.parent
else:
    raise RuntimeError(
        "Impossibile individuare la root della repository. "
        "Esegui il notebook dalla root del progetto oppure dalla cartella tests/."
    )

policies_path = repo_root / "config" / "policies.yaml"

assert policies_path.exists(), f"Policies config non trovato: {policies_path}"

print("Repository root:", repo_root)
print("Policies path:", policies_path)

Python interpreter: c:\Users\LMDP1\Desktop\GitHub\YgoMaster-PackBuilder\.venv\Scripts\python.exe
Current working directory: c:\Users\LMDP1\Desktop\GitHub\YgoMaster-PackBuilder\tests
Repository root: C:\Users\LMDP1\Desktop\GitHub\YgoMaster-PackBuilder
Policies path: C:\Users\LMDP1\Desktop\GitHub\YgoMaster-PackBuilder\config\policies.yaml


## 2. Import

In [13]:
from copy import deepcopy

from ygomaster_packbuilder.policies import load_policies_config

from ygomaster_packbuilder.validation import (
    ValidationIssue,
    ValidationReport,
    ValidationSeverity,
    validate_policies_config,
)

print("Policies validation imports OK")

Policies validation imports OK


## 3. Helper per visualizzare i report

In [14]:
def show_report(report: ValidationReport) -> None:
    print("Valid:", report.is_valid)
    print("Errors:", len(report.errors))
    print("Warnings:", len(report.warnings))
    print()

    for issue in report.issues:
        print(
            f"[{issue.severity.value.upper()}] "
            f"{issue.code} @ {issue.path}"
        )
        print(" ", issue.message)
        print()


def issue_codes(report: ValidationReport) -> list[str]:
    return [issue.code for issue in report.issues]


def assert_has_code(report: ValidationReport, code: str) -> None:
    assert code in issue_codes(report), (
        f"Expected validation code {code!r}, "
        f"found {issue_codes(report)!r}"
    )

print("Helpers ready")

Helpers ready


## 4. Caricamento della configurazione reale

In [15]:
config = load_policies_config(policies_path)

print("Loaded policies:", [policy.key for policy in config.policies])

Loaded policies: ['configuration-a']


## 5. Configurazione reale: validation

In [16]:
report = validate_policies_config(config)

show_report(report)

assert isinstance(report, ValidationReport)
assert report.is_valid
assert report.errors == []

print("Current policies config is valid")

Valid: True
Errors: 0
Warnings: 0

Current policies config is valid


## 6. Nessuna policy

In [17]:
case = deepcopy(config)
case.policies.clear()

report = validate_policies_config(case)

show_report(report)

assert not report.is_valid
assert_has_code(report, "no-policies")

Valid: False
Errors: 1
Warnings: 0

[ERROR] no-policies @ policies
  At least one opening policy must be defined.



## 7. Policy key vuota

In [18]:
case = deepcopy(config)

case.policies[0].key = ""

report = validate_policies_config(case)

show_report(report)

assert not report.is_valid
assert_has_code(report, "empty-policy-key")

Valid: False
Errors: 1
Warnings: 0

[ERROR] empty-policy-key @ policies[0].key
  Opening policy key cannot be empty.



## 8. Policy key duplicata

In [19]:
case = deepcopy(config)

duplicate = deepcopy(case.policies[0])
case.policies.append(duplicate)

report = validate_policies_config(case)

show_report(report)

assert not report.is_valid
assert_has_code(report, "duplicate-policy-key")

Valid: False
Errors: 1
Warnings: 0

[ERROR] duplicate-policy-key @ policies[1].key
  Opening policy key 'configuration-a' is defined more than once.



## 9. Probabilità negativa

In [20]:
case = deepcopy(config)

case.policies[0].normal_slot_distribution.normal = -1.0

report = validate_policies_config(case)

show_report(report)

assert not report.is_valid
assert_has_code(report, "probability-out-of-range")

Valid: False
Errors: 2
Warnings: 0

[ERROR] probability-out-of-range @ policies[0].normal_slot_distribution.normal
  Probability for 'normal' must be between 0 and 100.0.

[ERROR] invalid-probability-total @ policies[0].normal_slot_distribution
  Rarity distribution must sum to 100.0, but got 44.0.



## 10. Probabilità maggiore di 100

In [21]:
case = deepcopy(config)

case.policies[0].normal_slot_distribution.ultra_rare = 120.0

report = validate_policies_config(case)

show_report(report)

assert not report.is_valid
assert_has_code(report, "probability-out-of-range")

Valid: False
Errors: 2
Warnings: 0

[ERROR] probability-out-of-range @ policies[0].normal_slot_distribution.ultra_rare
  Probability for 'ultra_rare' must be between 0 and 100.0.

[ERROR] invalid-probability-total @ policies[0].normal_slot_distribution
  Rarity distribution must sum to 100.0, but got 217.5.



## 11. Distribuzione che non somma a 100

In [22]:
case = deepcopy(config)

distribution = case.policies[0].normal_slot_distribution
distribution.normal = 60.0
distribution.rare = 35.0
distribution.super_rare = 7.5
distribution.ultra_rare = 2.5

report = validate_policies_config(case)

show_report(report)

assert not report.is_valid
assert_has_code(report, "invalid-probability-total")

Valid: False
Errors: 1
Warnings: 0

[ERROR] invalid-probability-total @ policies[0].normal_slot_distribution
  Rarity distribution must sum to 100.0, but got 105.0.



## 12. Ten-pack guarantee attiva con distribuzione non valida

Quando `enabled=True`, la distribuzione finale deve essere validata.


In [23]:
case = deepcopy(config)

guarantee = case.policies[0].ten_pack_guarantee
guarantee.enabled = True
guarantee.final_slot_distribution.super_rare = 70.0
guarantee.final_slot_distribution.ultra_rare = 20.0

report = validate_policies_config(case)

show_report(report)

assert not report.is_valid
assert_has_code(report, "invalid-probability-total")

Valid: False
Errors: 1
Warnings: 0

[ERROR] invalid-probability-total @ policies[0].ten_pack_guarantee.final_slot_distribution
  Rarity distribution must sum to 100.0, but got 90.0.



## 13. Ten-pack guarantee disabilitata

Quando `enabled=False`, la distribuzione non viene interpretata semanticamente.


In [24]:
case = deepcopy(config)

guarantee = case.policies[0].ten_pack_guarantee
guarantee.enabled = False

guarantee.final_slot_distribution.normal = -999.0
guarantee.final_slot_distribution.rare = -999.0
guarantee.final_slot_distribution.super_rare = -999.0
guarantee.final_slot_distribution.ultra_rare = -999.0

report = validate_policies_config(case)

show_report(report)

assert "probability-out-of-range" not in issue_codes(report)
assert "invalid-probability-total" not in issue_codes(report)

print("Disabled ten-pack guarantee correctly ignored")

Valid: True
Errors: 0
Warnings: 0

Disabled ten-pack guarantee correctly ignored


## 14. UR pity attivo con distribuzione non valida

Una distribuzione attiva deve comunque sommare a 100.


In [25]:
case = deepcopy(config)

pity = case.policies[0].ultra_rare_pity
pity.enabled = True

pity.guaranteed_slot_distribution.ultra_rare = 90.0

report = validate_policies_config(case)

show_report(report)

assert not report.is_valid
assert_has_code(report, "invalid-probability-total")

Valid: False
Errors: 2
Warnings: 0

[ERROR] invalid-ur-pity-distribution @ policies[0].ultra_rare_pity.guaranteed_slot_distribution
  Enabled Ultra Rare pity must guarantee 100% ultra rare in its guaranteed slot distribution.

[ERROR] invalid-probability-total @ policies[0].ultra_rare_pity.guaranteed_slot_distribution
  Rarity distribution must sum to 100.0, but got 90.0.



## 15. UR pity attivo ma non 100% UR

Anche una distribuzione che somma correttamente a 100 deve essere rifiutata
se non garantisce realmente una Ultra Rare.


In [26]:
case = deepcopy(config)

pity = case.policies[0].ultra_rare_pity
pity.enabled = True

distribution = pity.guaranteed_slot_distribution
distribution.normal = 0.0
distribution.rare = 0.0
distribution.super_rare = 20.0
distribution.ultra_rare = 80.0

report = validate_policies_config(case)

show_report(report)

assert not report.is_valid
assert_has_code(report, "invalid-ur-pity-distribution")

Valid: False
Errors: 1
Warnings: 0

[ERROR] invalid-ur-pity-distribution @ policies[0].ultra_rare_pity.guaranteed_slot_distribution
  Enabled Ultra Rare pity must guarantee 100% ultra rare in its guaranteed slot distribution.



## 16. UR pity disabilitato

Quando `enabled=False`, la distribuzione del pity non viene interpretata semanticamente.


In [27]:
case = deepcopy(config)

pity = case.policies[0].ultra_rare_pity
pity.enabled = False

pity.guaranteed_slot_distribution.normal = -500.0
pity.guaranteed_slot_distribution.rare = -500.0
pity.guaranteed_slot_distribution.super_rare = -500.0
pity.guaranteed_slot_distribution.ultra_rare = -500.0

report = validate_policies_config(case)

show_report(report)

assert "invalid-ur-pity-distribution" not in issue_codes(report)

print("Disabled UR pity correctly ignored")

Valid: True
Errors: 0
Warnings: 0

Disabled UR pity correctly ignored


## 17. Raccolta di più errori contemporaneamente

Il validator deve continuare dopo il primo problema.


In [28]:
case = deepcopy(config)

policy = case.policies[0]

policy.key = ""
policy.normal_slot_distribution.normal = -5.0

policy.ten_pack_guarantee.enabled = True
policy.ten_pack_guarantee.final_slot_distribution.super_rare = 50.0
policy.ten_pack_guarantee.final_slot_distribution.ultra_rare = 20.0

policy.ultra_rare_pity.enabled = True
policy.ultra_rare_pity.guaranteed_slot_distribution.super_rare = 10.0
policy.ultra_rare_pity.guaranteed_slot_distribution.ultra_rare = 90.0

report = validate_policies_config(case)

show_report(report)

expected_codes = {
    "empty-policy-key",
    "probability-out-of-range",
    "invalid-probability-total",
    "invalid-ur-pity-distribution",
}

found_codes = set(issue_codes(report))
missing = expected_codes - found_codes

assert not missing, (
    f"Expected validation issues not found: {missing}. "
    f"Found: {found_codes}"
)

print("Multiple-error collection OK")

Valid: False
Errors: 5
Warnings: 0

[ERROR] empty-policy-key @ policies[0].key
  Opening policy key cannot be empty.

[ERROR] probability-out-of-range @ policies[0].normal_slot_distribution.normal
  Probability for 'normal' must be between 0 and 100.0.

[ERROR] invalid-probability-total @ policies[0].normal_slot_distribution
  Rarity distribution must sum to 100.0, but got 40.0.

[ERROR] invalid-probability-total @ policies[0].ten_pack_guarantee.final_slot_distribution
  Rarity distribution must sum to 100.0, but got 70.0.

[ERROR] invalid-ur-pity-distribution @ policies[0].ultra_rare_pity.guaranteed_slot_distribution
  Enabled Ultra Rare pity must guarantee 100% ultra rare in its guaranteed slot distribution.

Multiple-error collection OK


## 18. Controllo dei path degli errori

In [29]:
case = deepcopy(config)

case.policies[0].normal_slot_distribution.normal = -10.0

report = validate_policies_config(case)

matching = [
    issue
    for issue in report.issues
    if issue.code == "probability-out-of-range"
]

assert matching

issue = matching[0]

print("Code:", issue.code)
print("Path:", issue.path)
print("Message:", issue.message)

assert issue.path.endswith(".normal_slot_distribution.normal")

Code: probability-out-of-range
Path: policies[0].normal_slot_distribution.normal
Message: Probability for 'normal' must be between 0 and 100.0.
